# 加载模型

In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("正在加载模型（首次运行会下载约 500MB）...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


正在加载模型（首次运行会下载约 500MB）...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1028.28it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

# 查看模型内部向量表

In [2]:
embedding_table = model.transformer.wte.weight
print(embedding_table.shape)  # torch.Size([50257, 768])

torch.Size([50257, 768])


In [3]:
token_id = tokenizer.encode("France")[0]
vector = embedding_table[token_id]
print(vector.shape)                                # torch.Size([768])
print(vector[:16])                                  # 前 8 维: [-0.133, 0.098, 0.230, ...]

torch.Size([768])
tensor([-0.1329,  0.0977,  0.2297, -0.1049, -0.0112,  0.0470, -0.2165, -0.0211,
        -0.1277,  0.0466, -0.0395, -0.0616, -0.0606,  0.0109,  0.1844, -0.0117],
       grad_fn=<SliceBackward0>)


# 语义相似度

In [4]:
import torch

def cosine_similarity(v1, v2):
    return torch.dot(v1, v2) / (v1.norm() * v2.norm())

france = embedding_table[tokenizer.encode("France")[0]]
paris  = embedding_table[tokenizer.encode("Paris")[0]]
cat    = embedding_table[tokenizer.encode("cat")[0]]

sim1 = cosine_similarity(france, paris)  # ≈ 0.65  较高，有关系
sim2 = cosine_similarity(france, cat)    # ≈ 0.24  低，不相关
print(f"sim1 = {sim1}")
print(f"sim2 = {sim2}")

sim1 = 0.6504826545715332
sim2 = 0.24161531031131744


In [7]:
king = embedding_table[tokenizer.encode("king")[0]]
man = embedding_table[tokenizer.encode("man")[0]]
woman = embedding_table[tokenizer.encode("woman")[0]]
queen = embedding_table[tokenizer.encode("queen")[0]]
france = embedding_table[tokenizer.encode("France")[0]]
germany = embedding_table[tokenizer.encode("Germany")[0]]
paris = embedding_table[tokenizer.encode("Paris")[0]]
berlin = embedding_table[tokenizer.encode("Berlin")[0]]


print(f"king & queen = {cosine_similarity(king, queen)}")
change = king - man + woman
print(f"change & queen = {cosine_similarity(change, queen)}")

print(f"paris & berlin = {cosine_similarity(paris, berlin)}")
change = paris - france + germany
print(f"change & berlin = {cosine_similarity(change, berlin)}")

king & queen = 0.2666102647781372
change & queen = 0.2777973413467407
paris & berlin = 0.45116427540779114
change & berlin = 0.4842508137226105


# 比较任意词对的相似度

In [10]:
# ==================== 取出 Embedding 表 ====================

embedding_table = model.transformer.wte.weight.detach()
print(f"\nEmbedding 表的形状: {embedding_table.shape}")
print(f"  行数（词表大小）: {embedding_table.shape[0]}")
print(f"  列数（向量维度）: {embedding_table.shape[1]}")
print(f"  总参数量: {embedding_table.shape[0] * embedding_table.shape[1]:,}")

def get_vector(word):
    """获取一个词的 Embedding 向量，自动处理带/不带空格的变体"""
    for variant in [word, " " + word, word.lower(), " " + word.lower(),
                    word.capitalize(), " " + word.capitalize()]:
        ids = tokenizer.encode(variant)
        if len(ids) == 1:
            return embedding_table[ids[0]], ids[0], variant.strip()
    ids = tokenizer.encode(word)
    return embedding_table[ids[0]], ids[0], tokenizer.decode([ids[0]]).strip()

words = ["king", "man", "woman", "queen"]

# ==================== 展示向量 ====================

print(f"\n各 token 的向量（前 8 维）：")
print("-" * 70)

vectors = {}
for word in words:
    vec, tid, display = get_vector(word)
    vectors[word] = vec
    preview = [f"{v:.3f}" for v in vec[:8].tolist()]
    print(f"  '{display}' (ID {tid}): [{', '.join(preview)}, …]")

# ==================== 相似度矩阵 ====================

print(f"\n\n向量相似度（余弦相似度）：")
print("=" * 60)

computed_pairs = set()
results = []

for w1 in words:
    for w2 in words:
        if w1 >= w2:
            continue
        key = tuple(sorted([w1, w2]))
        if key in computed_pairs:
            continue
        computed_pairs.add(key)

        if w1 in vectors and w2 in vectors:
            sim = cosine_similarity(vectors[w1], vectors[w2]).item()
            results.append((w1, w2, sim))

# 按相似度降序排列
results.sort(key=lambda x: -x[2])

for w1, w2, sim in results:
    bar = "█" * int(max(0, sim) * 30)
    print(f"  '{w1}' vs '{w2}': {sim:>7.3f}  {bar}")

# ==================== 向量统计 ====================

print(f"\n\n向量统计信息：")
print("-" * 40)

for word in words[:5]:
    if word in vectors:
        vec = vectors[word]
        print(f"  '{word}': 范数={vec.norm().item():.3f}, "
              f"均值={vec.mean().item():.4f}, "
              f"标准差={vec.std().item():.4f}")


Embedding 表的形状: torch.Size([50257, 768])
  行数（词表大小）: 50257
  列数（向量维度）: 768
  总参数量: 38,597,376

各 token 的向量（前 8 维）：
----------------------------------------------------------------------
  'king' (ID 3364): [0.109, -0.395, 0.059, -0.102, -0.279, 0.260, -0.312, -0.088, …]
  'man' (ID 805): [-0.134, -0.138, 0.090, 0.065, -0.005, -0.011, -0.273, -0.018, …]
  'woman' (ID 8580): [-0.096, -0.077, 0.086, 0.251, -0.093, 0.065, -0.298, -0.116, …]
  'queen' (ID 16599): [-0.020, -0.290, -0.032, -0.059, 0.052, -0.083, -0.337, 0.015, …]


向量相似度（余弦相似度）：
  'man' vs 'woman':   0.586  █████████████████
  'queen' vs 'woman':   0.385  ███████████
  'king' vs 'woman':   0.294  ████████
  'king' vs 'man':   0.291  ████████
  'king' vs 'queen':   0.290  ████████
  'man' vs 'queen':   0.217  ██████


向量统计信息：
----------------------------------------
  'king': 范数=4.152, 均值=0.0016, 标准差=0.1499
  'man': 范数=3.564, 均值=-0.0009, 标准差=0.1287
  'woman': 范数=4.113, 均值=0.0021, 标准差=0.1485
  'queen': 范数=3.785, 均值=-0.0000, 标准

# 多义词

In [ ]:
import torch

def get_all_layer_vectors(text, target_word):
    """获取目标词在每一层 Transformer 输出的向量"""
    input_ids = tokenizer.encode(text, return_tensors="pt")
    print(input_ids)
    tokens = [tokenizer.decode([id]) for id in input_ids[0]]
    print(tokens)

    # 找到目标词的位置
    target_pos = None
    for i, t in enumerate(tokens):
        if target_word.lower() in t.strip().lower():
            target_pos = i
            break

    if target_pos is None:
        return None, tokens, None

    with torch.no_grad():
        outputs = model(input_ids, output_hidden_states=True)
        print(outputs.hidden_states)

    # hidden_states[0] = Embedding 层, [1]-[12] = Transformer 各层
    layer_vectors = [hs[0, target_pos, :].detach() for hs in outputs.hidden_states]

    return layer_vectors, tokens, target_pos

get_all_layer_vectors("I deposited money at the bank today", "bank")

tensor([[   40, 27163,  1637,   379,   262,  3331,  1909]])
['I', ' deposited', ' money', ' at', ' the', ' bank', ' today']
CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[ -39.3084,  -39.0100,  -41.8374,  ...,  -46.9337,  -44.9074,
           -39.5149],
         [-101.7301, -101.0114, -106.4455,  ..., -106.9139, -110.6890,
          -104.0121],
         [-134.3702, -133.6237, -141.1262,  ..., -144.0326, -146.3539,
          -137.5739],
         ...,
         [ -98.7983,  -97.2641, -100.3773,  ...,  -99.2126, -105.5087,
           -97.6603],
         [-118.9466, -118.3095, -124.3675,  ..., -127.0715, -131.4665,
          -121.9434],
         [-104.5926, -105.2584, -111.0728,  ..., -119.8575, -117.8034,
          -107.8066]]]), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=(tensor([[[ 0.1286, -0.2933,

([tensor([ 5.5309e-02,  4.7965e-02,  2.6869e-01,  3.4289e-02, -3.7104e-02,
           7.1834e-02, -3.2842e-01, -1.4488e-01,  7.1252e-02,  2.2543e-01,
          -1.5278e-01,  1.2111e-02, -1.7007e-01, -2.5271e-02,  3.5696e-03,
           1.2754e-01,  8.7374e-02, -8.9331e-02, -2.2160e-01,  4.9682e-01,
           2.0423e-01,  9.5232e-02, -2.5768e-02, -5.1335e-02, -3.7263e-02,
           1.1094e-01,  2.5069e-01,  1.4676e-01, -1.2149e-02, -3.4562e-02,
          -2.3136e-01, -4.2062e-03,  8.1714e-03, -2.2905e-01, -1.5081e-01,
           1.8939e-01, -5.7185e-01,  8.7380e-02, -3.7482e-02, -1.8761e-01,
           1.1373e-01,  1.1964e-01, -1.0632e-01,  9.0264e-02, -5.0503e-02,
           3.3953e-02, -9.5023e-02, -2.4703e-02, -1.1578e-02, -5.2987e-01,
           5.6411e-02, -5.1036e-02,  4.7787e-01, -2.1012e-01,  2.1779e-02,
          -3.2461e-01,  1.0640e-01, -8.7275e-02,  9.3187e-02,  3.9756e-02,
           1.9701e-01,  6.5332e-02,  3.6751e-02,  6.3956e-02,  5.4233e-02,
           1.3743e-01, -2

# 位置编码

In [8]:
position_table = model.transformer.wpe.weight
print(position_table.shape)  # torch.Size([1024, 768])
# 1024 个位置，每个位置 768 维 → GPT-2 最多处理 1024 个 token

torch.Size([1024, 768])
